In [ ]:
import polars as pl
import zh

data = pl.read_parquet("../data/1m/1m.parquet").lazy()

目前发现：可以考虑使用maker单节省手续费，但必须考虑挂单方向导致的成交概率问题和价格跳动问题，已知ETHUSDT的最小变动单位为0.01，价格为3000左右，可以假定手续费至少0.1bps。在这一假设下，原先的配对交易完全失效，考虑单边类布林带策略。

In [ ]:
def backtest_pairs_trading(
    data: pl.LazyFrame,
    symbol1: str,
    symbol2: str,
    start_date,
    end_date,
    entry_z: float = 2.5,
    fee_bps: float = 10,
    window_size: int = 120,
):
    """
    Backtest a pairs trading strategy on two symbols.

    Parameters:
    -----------
    data : polars.LazyFrame
        Price data with columns: symbol, open_time, close
    symbol1 : str
        First symbol (e.g., 'BTCUSDT')
    symbol2 : str
        Second symbol (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.5)
    fee_bps : float
        Fee in basis points per leg (default: 0.2)
    window_size : int
        Rolling window size for calculations (default: 120)
    Returns:
    --------
    dict with keys: 'backtest', 'metrics', 'data'
    """

    fee_rate = fee_bps / 1e4

    # Prepare data
    data_combined = (
        data.filter(pl.col("symbol") == symbol1)
        .select(
            "open_time",
            pl.col("close").alias(f"{symbol1}_price"),
            pl.col("close").log().alias("S1"),
        )
        .join(
            data.filter(pl.col("symbol") == symbol2).select(
                "open_time",
                pl.col("close").alias(f"{symbol2}_price"),
                pl.col("close").log().alias("S2"),
            ),
            on="open_time",
        )
    ).filter((pl.col("open_time") >= start_date) & (pl.col("open_time") < end_date))

    # Backtest
    bt = (
        data_combined.sort("open_time")
        .with_columns(
            pl.rolling_cov("S1", "S2", window_size=window_size).alias("rolling_cov"),
            pl.col("S2").rolling_var(window_size=window_size).alias("rolling_var_s2"),
        )
        .with_columns(
            (pl.col("rolling_cov") / pl.col("rolling_var_s2")).alias("rolling_beta"),
        )
        .with_columns(
            (
                pl.col("S1").rolling_mean(window_size=window_size)
                - pl.col("rolling_beta")
                * pl.col("S2").rolling_mean(window_size=window_size)
            ).alias("rolling_alpha"),
        )
        .with_columns(
            (
                pl.col("S1")
                - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("S2"))
            ).alias("rolling_residual"),
        )
        .with_columns(
            (
                (
                    pl.col("rolling_residual")
                    - pl.col("rolling_residual").rolling_mean(window_size=window_size)
                )
                / pl.col("rolling_residual").rolling_std(window_size=window_size)
            ).alias("rolling_zscore"),
        )
        .with_columns(
            pl.col(f"{symbol1}_price").pct_change().alias("r_s1"),
            pl.col(f"{symbol2}_price").pct_change().alias("r_s2"),
            pl.col("rolling_beta").shift(1).alias("beta_lag"),
            pl.col("rolling_zscore").shift(1).alias("z_lag"),
        )
        .filter(pl.col("r_s2").is_not_null())
        .with_columns(
            pl.when(pl.col("z_lag") >= entry_z)
            .then(-1)
            .when(pl.col("z_lag") <= -entry_z)
            .then(1)
            .otherwise(0)
            .alias("pos")
        )
        .with_columns(
            (pl.col("r_s1") - pl.col("beta_lag") * pl.col("r_s2")).alias("r_spread_raw"),
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            # Capital-weighted PnL: normalize by (1 + |beta|) to account for unequal capital allocation
            (
                pl.col("pos") * pl.col("r_spread_raw") / (1 + pl.col("beta_lag").abs())
            ).alias("pnl_gross"),
            (
                (pl.col("pos") - pl.col("pos_prev")).abs()
                * (1 + pl.col("beta_lag").abs())
            ).alias("turnover_units"),
        )
        .with_columns((fee_rate * pl.col("turnover_units")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("pnl_net"))
        .with_columns(
            (1 + pl.col("pnl_net")).cum_prod().alias("equity"),
            pl.col("pnl_net").cum_sum().alias("cum_return"),
        )
        .drop_nulls()
    )

    out = bt.collect()



    return out

In [ ]:
zh.metrics(backtest_pairs_trading(
            data,
            symbol1="BTCUSDT",
            symbol2="ETHUSDT",
            start_date=pl.datetime(2025, 1, 1),
            end_date=pl.datetime(2025, 12, 31),
            entry_z=2.0,
            window_size=30,
            fee_bps=0.0,
        )
        .group_by_dynamic(pl.col("open_time"), every="1d")
    .agg(pl.col("pnl_net").sum().alias("portfolio_return"))
).collect(engine="streaming")

In [ ]:
backtest_pairs_trading(
            data,
            symbol1="BTCUSDT",
            symbol2="ETHUSDT",
            start_date=pl.datetime(2025, 1, 1),
            end_date=pl.datetime(2025, 12, 31),
            entry_z=2.0,
            window_size=30,
            fee_bps=0.0,
        ).group_by_dynamic(pl.col("open_time"), every="1d").agg(
    pl.col("pnl_net").sum()
).with_columns(pl.col("pnl_net").cum_sum()).plot.line(
    x="open_time", y="pnl_net"
)

In [ ]:
def backtest_single_asset(
    data: pl.LazyFrame,
    symbol: str,
    start_date,
    end_date,
    entry_z: float = 2.0,
    fee_bps: float = 0.0,
    window_size: int = 14,
)-> pl.LazyFrame:
    """
    Backtest a mean reversion strategy on a single asset with 1-minute hold period.

    Parameters:
    -----------
    data : polars.LazyFrame
        Price data with columns: symbol, open_time, close
    symbol : str
        Symbol to trade (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.0)
    fee_bps : float
        Fee in basis points (default: 0)
    window_size : int
        Rolling window size for calculations (default: 14)
    """

    fee_rate = fee_bps / 1e4

    # Prepare data
    asset_data = (
        data.filter(pl.col("symbol") == symbol)
        .select(
            "open_time",
            pl.col("close").alias("price"),
            pl.col("close").log().alias("log_price"),
        )
        .filter((pl.col("open_time") >= start_date) & (pl.col("open_time") <= end_date))
        .sort("open_time")
    )

    # Backtest
    bt = (
        asset_data.with_columns(
            pl.col("log_price")
            .rolling_mean(window_size=window_size)
            .shift(1)
            .alias("rolling_mean"),
            pl.col("log_price")
            .rolling_std(window_size=window_size)
            .shift(1)
            .alias("rolling_std"),
        )
        .with_columns(
            (
                (pl.col("log_price") - pl.col("rolling_mean")) / pl.col("rolling_std")
            ).alias("zscore")
        )
        .with_columns(
            pl.col("price").pct_change().alias("returns"),
            pl.col("zscore").shift(1).alias("zscore_lag"),
        )
        .filter(pl.col("returns").is_not_null())
        .with_columns(
            # Mean reversion: buy when zscore is low (oversold), sell when high (overbought)
            pl.when(pl.col("zscore_lag") <= -entry_z)
            .then(1)  # Long when oversold
            .when(pl.col("zscore_lag") >= entry_z)
            .then(-1)  # Short when overbought
            .otherwise(0)
            .alias("pos")
        )
        .with_columns(
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            (pl.col("pos") * pl.col("returns")).alias("pnl_gross"),
            (pl.col("pos") - pl.col("pos_prev")).abs().alias("turnover"),
        )
        .with_columns((fee_rate * pl.col("turnover")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("pnl_net"))
        .fill_null(0)
    )

    return bt

In [ ]:
# Test pure ETH strategy
backtest_single_asset(
    data,
    symbol="ETHUSDT",
    start_date=pl.datetime(2025, 1, 1),
    end_date=pl.datetime(2025, 12, 31),
    entry_z=2.0,
    window_size=14,
    fee_bps=0.0,
).group_by_dynamic(pl.col("open_time"), every="1d").agg(
    pl.col("pnl_net").sum()
).with_columns(pl.col("pnl_net").cum_sum()).collect(engine="streaming").plot.line(
    x="open_time", y="pnl_net"
)

In [ ]:


zh.metrics(  # Test pure ETH strategy
    backtest_single_asset(
        data,
        symbol="ETHUSDT",
        start_date=pl.datetime(2025, 1, 1),
        end_date=pl.datetime(2025, 12, 31),
        entry_z=2,
        window_size=14,
        fee_bps=0.0,
    )
    .group_by_dynamic(pl.col("open_time"), every="1d")
    .agg(pl.col("pnl_net").sum().alias("portfolio_return"))
).collect(engine="streaming")